# Código atualizado


In [1]:
# ============================================
# 0) Imports
# ============================================
import re
import os
import json
import warnings

import numpy as np
import pandas as pd
import numpy as np

from dataclasses import dataclass
from typing import Callable, List, Tuple, Dict, Any, Optional
from datetime import datetime

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from itertools import combinations
from sklearn.preprocessing import PolynomialFeatures

import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


In [2]:
# ============================================
# 1) Carregar datasets
# ============================================
DATASET_HC = 'dataset_voz_completo_HC.csv'
DATASET_PD = 'dataset_voz_completo_PD.csv'

df_HC = pd.read_csv(DATASET_HC)
df_PD = pd.read_csv(DATASET_PD)
print("Shape original HC:", df_HC.shape)
print("Shape original DF:", df_PD.shape)

df_HC['status'] = 0
df_PD['status'] = 1

df_concat = pd.concat([df_HC, df_PD], ignore_index=True)

df_concat.to_csv('dataset_concatenado.csv', index=False)

print("Shape final:", df_concat.shape)
print(df_concat)
df_concat.head()


Shape original HC: (41, 88)
Shape original DF: (40, 88)
Shape final: (81, 89)
                                            file_name  group  f0_mean_hz  \
0    AH_064F_7AB034C9-72E4-438B-A9B3-AD7FDA1596C5.wav  HC_AH  131.121658   
1    AH_114S_A89F3548-0B61-4770-B800-2E26AB3908B6.wav  HC_AH  110.595296   
2    AH_121A_BD5BA248-E807-4CB9-8B53-47E7FFE5F8E2.wav  HC_AH  234.182987   
3    AH_123G_559F0706-2238-447C-BA39-DB5933BA619D.wav  HC_AH  102.822765   
4    AH_195B_39DA6A45-F4CC-492A-80D4-FB79049ACC22.wav  HC_AH  111.466057   
..                                                ...    ...         ...   
76  AH_545841223-24FB0419-5BAE-4F9C-8EBC-CD62DA659...  PD_AH  197.627000   
77  AH_545841226-C699FC9E-1E0C-474D-A12A-936DD92B8...  PD_AH  170.152161   
78  AH_545841227-5C77713A-66F1-49D0-BC8A-702C152E6...  PD_AH  258.527907   
79  AH_545847410-D1BA3BB4-1F61-44CA-ACDE-455A8E97E...  PD_AH  125.287455   
80  AH_545880204-EE87D3E2-0D4C-4EAA-ACD7-C3F177AFF...  PD_AH  112.002244   

    f0_st

,file_name,group,f0_mean_hz,f0_std_hz,f0_min_hz,f0_max_hz,f0_cv,f1_mean_hz,f1_std_hz,f2_mean_hz,...,spec_flux_mean,spec_flux_std,spec_energy_low_mean,spec_energy_mid_mean,spec_energy_high_mean,tsallis_sq_amp,shannon_s1_amp,tsallis_sq_f0,shannon_s1_f0,status
0,AH_064F_7AB034C9-72E4-438B-A9B3-AD7FDA1596C5.wav,HC_AH,131.121658,1.382661,127.979975,135.285714,0.010545,670.176352,11.476255,1144.639964,...,0.113668,0.042128,2.135428,12.731040,0.035838,2.175084,3.616453,0.471591,0.543912,0
1,AH_114S_A89F3548-0B61-4770-B800-2E26AB3908B6.wav,HC_AH,110.595296,1.771228,105.192754,115.377726,0.016015,624.156806,50.759496,975.646622,...,0.183278,0.084410,6.280130,4.587857,0.013371,2.150822,3.590799,0.619451,0.727357,0
2,AH_121A_BD5BA248-E807-4CB9-8B53-47E7FFE5F8E2.wav,HC_AH,234.182987,1.199626,231.386563,237.600742,0.005123,475.504882,48.768486,883.969002,...,0.076905,0.025209,34.665394,8.976105,0.020312,2.141979,3.551923,0.000000,-0.000000,0
3,AH_123G_559F0706-2238-447C-BA39-DB5933BA619D.wav,HC_AH,102.822765,0.989926,100.932166,106.161438,0.009628,595.806684,165.885054,1011.652389,...,0.115013,0.034062,16.381947,10.330296,0.079708,2.144335,3.571178,0.000000,-0.000000,0
4,AH_195B_39DA6A45-F4CC-492A-80D4-FB79049ACC22.wav,HC_AH,111.466057,1.052937,109.405717,114.119047,0.009446,698.353948,45.373953,1107.937845,...,0.178199,0.063370,1.263110,4.190388,0.049585,2.157088,3.593633,0.285120,0.352543,0


In [4]:
# ============================================
# 2) Identificar colunas com valores nulos
# ============================================
cols_with_nans = df_concat.columns[df_concat.isnull().any()].tolist()
print(cols_with_nans)

# Configurar e aplicar o Imputador pela Mediana
# A mediana é mais robusta a outliers comuns em sinais de áudio
imputer = SimpleImputer(strategy='median')

# Criamos uma cópia para preservar o dataframe original se necessário
df_imputed = df_concat.copy()

# Aplicamos a transformação apenas nas colunas identificadas
df_imputed[cols_with_nans] = imputer.fit_transform(df_imputed[cols_with_nans])
df_imputed.to_csv('dataset_imputado.csv', index=False)

# 4. Verificação
print(f"Imputação concluída nas colunas: {cols_with_nans}")
print("Total de valores nulos no dataset após o processo:", df_imputed.isnull().sum().sum())

['jitter_local', 'jitter_rap', 'jitter_ppq5', 'tsallis_sq_f0', 'shannon_s1_f0']
Imputação concluída nas colunas: ['jitter_local', 'jitter_rap', 'jitter_ppq5', 'tsallis_sq_f0', 'shannon_s1_f0']
Total de valores nulos no dataset após o processo: 0


In [5]:
# ============================================
# 3) Extração do subject_id
# Padrão: AH_064F_UUID.wav -> Extrai '064F'
# ============================================
def extract_subject_id(name):
    parts = str(name).split('_')
    return parts[1] if len(parts) >= 2 else None

df = pd.read_csv('dataset_imputado.csv')
df["subject_id"] = df["file_name"].apply(extract_subject_id)

# checagem rápida
if df["subject_id"].isna().any():
    raise ValueError("Falha ao extrair subject_id de algumas linhas. Verifique o padrão da coluna 'name'.")

print("Nº de sujeitos:", df["subject_id"].nunique())

Nº de sujeitos: 81


In [6]:
# ============================================
# 4) Preparar X, y e grupos
# Removemos metadados e o target das features
# ============================================
cols_to_drop = ["file_name", "group", "status", "subject_id"]
feature_cols = [c for c in df.columns if c not in cols_to_drop]

X = df[feature_cols]
y = df["status"].astype(int)
groups = df["subject_id"]

print("\nBalanceamento (após limpeza):")
print(pd.Series(y).value_counts().rename(index={0: "Controle(0)", 1: "Parkinson(1)"}))


Balanceamento (após limpeza):
status
Controle(0)     41
Parkinson(1)    40
Name: count, dtype: int64


In [7]:
# ============================================
# 5) Split Treino/Teste SEM leakage (por sujeito) 80/20
#    (apenas cria os índices; não treina nada aqui)
# ============================================
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

# verificando a integridade
train_subjects = set(groups[train_idx])
test_subjects = set(groups[test_idx])
intersection = train_subjects.intersection(test_subjects)

print(f"Sujeitos no Treino: {len(train_subjects)} | Teste: {len(test_subjects)}")
print(f"Leakage check (deve ser 0): {len(intersection)} interseções encontradas.")
if len(intersection) != 0:
    raise ValueError("Houve leakage: mesmos sujeitos em treino e teste.")

Sujeitos no Treino: 64 | Teste: 17
Leakage check (deve ser 0): 0 interseções encontradas.


In [8]:
# ============================================
# 6) QuantileClipper
# ============================================
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, low=0.01, high=0.99):
        self.low = low
        self.high = high

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        # Calcula os limites (quantis) para cada coluna
        self.lo_ = np.nanquantile(X, self.low, axis=0)
        self.hi_ = np.nanquantile(X, self.high, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        # Aplica o limitador: valores abaixo de lo_ viram lo_ 
        # e acima de hi_ viram hi_
        return np.clip(X, self.lo_, self.hi_)

In [9]:
# ============================================ VERIFICAR A RETIRADA




# 7) Pipeline base completo (com SMOTE) - pronto para uso posterior
#    IMPORTANTE: ainda NÃO há fit/cv aqui.
# ============================================
preprocess_and_smote = ImbPipeline(steps=[
    ("clip", QuantileClipper(0.01, 0.99)),
    ("scale", RobustScaler()),
    ("smote", SMOTE(random_state=42, k_neighbors=3)),
])
# Este pipeline é "base". Depois acoplaremos um modelo:
# final_pipeline = ImbPipeline(steps=[("prep", preprocess_and_smote), ("clf", SEU_MODELO)])


In [10]:
# ============================================
# 8) Função: consenso de features
# ============================================
def get_consensus_features(rf_imp, xgb_imp, cat_imp, top_k=16):
    """Combine rankings of the 3 models to obtain consensus on the best features"""

    rf_ranked = rf_imp.reset_index(drop=True)
    xgb_ranked = xgb_imp.reset_index(drop=True)
    cat_ranked = cat_imp.reset_index(drop=True)

    rf_rank_dict = {row['feature']: idx + 1 for idx, row in rf_ranked.iterrows()}
    xgb_rank_dict = {row['feature']: idx + 1 for idx, row in xgb_ranked.iterrows()}
    cat_rank_dict = {row['feature']: idx + 1 for idx, row in cat_ranked.iterrows()}

    all_features = set(rf_rank_dict.keys()) | set(xgb_rank_dict.keys()) | set(cat_rank_dict.keys())

    consensus_ranking = {}
    for feature in all_features:
        rf_pos = rf_rank_dict.get(feature, len(rf_ranked) + 1)
        xgb_pos = xgb_rank_dict.get(feature, len(xgb_ranked) + 1)
        cat_pos = cat_rank_dict.get(feature, len(cat_ranked) + 1)
        consensus_ranking[feature] = (rf_pos + xgb_pos + cat_pos) / 3

    sorted_features = sorted(consensus_ranking.items(), key=lambda x: x[1])
    return [feature for feature, _ in sorted_features[:top_k]]


In [11]:
# Pré-processamento comum
preprocess = ImbPipeline(steps=[
    ("clip", QuantileClipper(0.01, 0.99)),
    ("scale", RobustScaler())
])

X_proc = preprocess.fit_transform(X)


In [12]:
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_proc, y)

rf_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)


In [15]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_proc, y)

xgb_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb.feature_importances_
}).sort_values("importance", ascending=False)


In [16]:
cat = CatBoostClassifier(
    iterations=500,
    depth=5,
    learning_rate=0.05,
    loss_function="Logloss",
    verbose=False,
    random_seed=42
)

cat.fit(X_proc, y)

cat_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": cat.get_feature_importance()
}).sort_values("importance", ascending=False)


In [17]:
top_features = get_consensus_features(
    rf_imp=rf_importance,
    xgb_imp=xgb_importance,
    cat_imp=cat_importance,
    top_k=16
)

print("Top 16 features por consenso:")
for f in top_features:
    print("-", f)


Top 16 features por consenso:
- mfcc11_std
- mfcc13_std
- mfcc1_std
- mfcc10_std
- dmfcc3_mean
- dmfcc13_std
- tsallis_sq_amp
- mfcc9_std
- shimmer_local
- mfcc8_std
- mfcc12_std
- spec_rolloff_std_hz
- mfcc1_mean
- dmfcc13_mean
- dmfcc7_std
- shimmer_apq3


In [ ]:
# ============================================
# 9) Definir as features selecionadas (Top-11)
# ============================================
selected_features = [
    "",""
]

missing = set(selected_features) - set(df_real.columns)
if missing:
    raise ValueError(f"Colunas selecionadas não encontradas no dataset: {missing}")


In [ ]:
# ============================================
# 10) Criar dataset reduzido (features + status + subject_id)
# ============================================
df_reduced = df_real[selected_features + ["status", "subject_id"]].copy()
print("\nShape dataset reduzido:", df_reduced.shape)


In [ ]:
# ============================================
# 11) Salvar datasets (limpo e reduzido)
# ============================================
clean_path = "dataset_clean_no_synth.csv"
reduced_path = "dataset_reduced_top11.csv"

df_real.to_csv(clean_path, index=False)
df_reduced.to_csv(reduced_path, index=False)

print("\nArquivos salvos:")
print("-", clean_path)
print("-", reduced_path)

In [ ]:
def make_pipeline(model):
    return ImbPipeline(steps=[
        # 1. Imputação: Garante que não haja nulos entrando nos transformadores
        ("imputer", SimpleImputer(strategy='median')),

        # 2. Clipping: Amortece os outliers extremos antes do escalonamento
        ("clip", QuantileClipper(0.01, 0.99)),
        
        # 3. RobustScaler: Escalonamento robusto baseado em quartis
        ("scale", RobustScaler()),

        # 4. SMOTE: Balanceamento sintético aplicado apenas durante o 'fit' (treino)
        ("smote", SMOTE(random_state=42, k_neighbors=3)),

        # 5. O Classificador (SVC, XGBoost, etc.)
        ("clf", model)
    ])
    

In [ ]:
# =========================================================
# 13) Benchmark: roda GroupKFold CV e devolve tabela resumo
# =========================================================
def benchmark_models(X, y, groups, models_dict, n_splits=5, n_jobs=1):
    cv = GroupKFold(n_splits=n_splits)

    scoring = {
        "bal_acc": "balanced_accuracy",
        "roc_auc": "roc_auc",
        "f1": "f1",
    }

    rows = []
    for name, model in models_dict.items():
        pipe = make_pipeline(model)

        scores = cross_validate(
            pipe, X, y,
            groups=groups,
            cv=cv,
            scoring=scoring,
            n_jobs=n_jobs,
            error_score="raise"
        )

        rows.append({
            "model": name,
            "bal_acc_mean": float(np.mean(scores["test_bal_acc"])),
            "bal_acc_std":  float(np.std(scores["test_bal_acc"])),
            "roc_auc_mean": float(np.mean(scores["test_roc_auc"])),
            "roc_auc_std":  float(np.std(scores["test_roc_auc"])),
            "f1_mean":      float(np.mean(scores["test_f1"])),
            "f1_std":       float(np.std(scores["test_f1"])),
        })

    return pd.DataFrame(rows).sort_values("bal_acc_mean", ascending=False).reset_index(drop=True)


In [ ]:
# =========================================================
# 14) Carregar dataset para benchmarking
#    Escolha UM:
#    A) dataset reduzido Top-11 (recomendado pelo tamanho do dataset)
#    B) dataset completo limpo (sem synth)
# =========================================================

USE_REDUCED = True  # mude para False se quiser testar com dataset completo

if USE_REDUCED:
    df = pd.read_csv("dataset_reduced_top11.csv")
    feature_cols = [c for c in df.columns if c not in ["status", "subject_id"]]
else:
    df = pd.read_csv("/dataset_clean_no_synth.csv")
    feature_cols = [c for c in df.columns if c not in ["name", "status", "subject_id"]]

X = df[feature_cols].values
y = df["status"].astype(int).values
groups = df["subject_id"].values

print("Dataset usado:", "REDUZIDO Top-11" if USE_REDUCED else "COMPLETO (limpo)")
print("Shape:", df.shape)
print("Nº features:", len(feature_cols))
print("Nº sujeitos:", df["subject_id"].nunique())
print("Balanceamento:", pd.Series(y).value_counts().to_dict())

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
# =========================================================
# 15) Modelos (SEM CatBoost)
# =========================================================
RNG_SEED = 42

models = {
    "logreg_l2": LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            max_iter=5000,
            class_weight="balanced",
            random_state=RNG_SEED,
            fit_intercept=True,
        ),
    "svc_rbf":  SVC(
            C=100.0,
            gamma=0.1,
            kernel="rbf",
            probability=False,          # necessário para ROC-AUC via predict_proba
            class_weight="balanced",
            random_state=RNG_SEED,      # usado na calibração interna do probability=True
        ),
    # LinearSVC não tem predict_proba -> calibramos para ter probabilidade para ROC-AUC
    "linear_svc_cal": CalibratedClassifierCV(
            estimator=LinearSVC(
                C=1.0,
                class_weight="balanced",
                random_state=RNG_SEED,
                max_iter=10000,
            ),
            method="sigmoid",
            cv=3,
        ),
    # -------------------------
    # Árvores / Ensembles
    # -------------------------
    "random_forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=5,
            max_features="sqrt",
            bootstrap=True,
            class_weight="balanced",
            random_state=RNG_SEED,
            n_jobs=-1,
        ),
    "gradient_boosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            subsample=1.0,
            random_state=RNG_SEED,
        ),
    "adaboost": AdaBoostClassifier(
            n_estimators=100,
            learning_rate=1.0,            
            random_state=RNG_SEED,
        ),
    "decision_tree": DecisionTreeClassifier(
            criterion="gini",
            max_depth=5,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=RNG_SEED,
        ),
    # -------------------------
    # KNN (não tem random_state)
    # -------------------------
    "knn": KNeighborsClassifier(
            n_neighbors=7,
            weights="distance",
            metric="minkowski",
            p=2,
        ),
    
    # -------------------------
    # MLP (rede rasa)
    # -------------------------
    "mlp": MLPClassifier(
            hidden_layer_sizes=(50, 50),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            learning_rate="constant",
            learning_rate_init=0.001,
            max_iter=2000,
            tol=1e-5,
            early_stopping=False,
            random_state=RNG_SEED,
        ),
    # -------------------------
    # XGBoost / CatBoost
    # -------------------------
    "xgboost": XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=1.0,
            reg_lambda=1.0,
            reg_alpha=0.0,
            gamma=0.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RNG_SEED,
            n_jobs=-1,
            verbosity=0,
        ),

}


In [ ]:
# =========================================================
# 16) Rodar benchmark (GroupKFold por sujeito)
# =========================================================
results = benchmark_models(X, y, groups, models, n_splits=5, n_jobs=1)

# Mostrar ranking
print("\n=== Ranking por Balanced Accuracy (GroupKFold, sem leakage) ===")
print(results)

In [ ]:
# Modelo final para serialização (pipeline + treino + save)

import numpy as np
import joblib

from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# sua classe já existente
# from your_module import QuantileClipper


def build_final_svc_pipeline(
    C: float,
    gamma: float,
    rng_seed: int = 42,
    smote_k_neighbors: int = 3,
) -> ImbPipeline:
    """
    Pipeline final (treino e inferência):
        clip -> robustscale -> SMOTE (somente no fit) -> SVC(RBF)

    Observação: imblearn Pipeline aplica SMOTE apenas no fit; no predict o sampler é ignorado.
    """
    clf = SVC(
        C=float(C),
        gamma=float(gamma),
        kernel="rbf",
        probability=False,          # mais rápido; usa decision_function
        class_weight="balanced",
        random_state=rng_seed,
    )

    pipe = ImbPipeline(steps=[
        ("clip", QuantileClipper(0.01, 0.99)),
        ("scale", RobustScaler()),
        ("smote", SMOTE(random_state=rng_seed, k_neighbors=smote_k_neighbors)),
        ("clf", clf),
    ])
    return pipe


def fit_and_serialize_final_model(
    X,
    y,
    C: float,
    gamma: float,
    out_path: str = "final_svc_rbf_pipeline.joblib",
    rng_seed: int = 42,
    smote_k_neighbors: int = 3,
):
    """
    Treina no dataset inteiro (Top-11 ou completo, você decide antes) e salva o pipeline.
    """
    pipe = build_final_svc_pipeline(
        C=C,
        gamma=gamma,
        rng_seed=rng_seed,
        smote_k_neighbors=smote_k_neighbors,
    )
    pipe.fit(X, y)
    joblib.dump(
        {
            "pipeline": pipe,
            "meta": {
                "model": "SVC_RBF",
                "C": float(C),
                "gamma": float(gamma),
                "rng_seed": int(rng_seed),
                "smote_k_neighbors": int(smote_k_neighbors),
                "features_shape": getattr(X, "shape", None),
            },
        },
        out_path,
    )
    return out_path


def fit_and_serialize_final_model_top11(
    X, y,
    C_star: float,
    gamma_star: float,
    t_star: float,
    out_path: str = "final_svc_rbf_top11.joblib",
    rng_seed: int = 42,
    smote_k_neighbors: int = 3,
):
    import joblib
    import numpy as np

    t_star = float(np.clip(t_star, 0.10, 0.90))

    pipe = build_final_svc_pipeline(
        C=C_star,
        gamma=gamma_star,
        rng_seed=rng_seed,
        smote_k_neighbors=smote_k_neighbors,
    )
    pipe.fit(X, y)

    payload = {
        "pipeline": pipe,
        "threshold_t": t_star,
        "meta": {
            "dataset": "dataset_reduced_top11.csv",
            "model": "SVC_RBF",
            "C": float(C_star),
            "gamma": float(gamma_star),
            "threshold_t": float(t_star),
            "rng_seed": int(rng_seed),
            "smote_k_neighbors": int(smote_k_neighbors),
            "n_features": int(X.shape[1]),
            "n_samples": int(X.shape[0]),
        },
    }
    joblib.dump(payload, out_path)
    return out_path


# Passo 1: Implementação do MOPSO (Multi-Objective PSO)

- ✅ Entradas: X, y, groups (Top-11), eval_fn (sua função), e parâmetros do MOPSO
- ✅ Saídas: archive (Pareto final) + best_choice (solução selecionada por “topo + estabilidade”)

In [ ]:
# Função de avaliação do PSO (3 parâmetros, 2 objetivos)
"""
Aqui está a função “fitness” do PSO baseada no seu pipeline, mas adaptada para otimizar threshold 
corretamente por fold e evitar falhas do SMOTE quando houver poucos exemplos da classe minoritária 
em algum fold.
"""

def _safe_smote_k(y_train, desired_k=3) -> int:
    """
    Garante que o SMOTE não quebre quando a classe minoritária tem poucos exemplos no fold.
    Regra: k_neighbors <= (n_minority - 1), e no mínimo 1.
    """
    # conta classe minoritária
    vals, counts = np.unique(y_train, return_counts=True)
    n_minority = int(np.min(counts))
    k = min(int(desired_k), max(1, n_minority - 1))
    return k


def _make_fold_pipeline_svc(C, gamma, rng_seed=42, smote_k_neighbors=3) -> ImbPipeline:
    """
    Cria pipeline para o fold (permite ajustar k_neighbors do SMOTE por fold).
    """
    clf = SVC(
        C=float(C),
        gamma=float(gamma),
        kernel="rbf",
        probability=False,
        class_weight="balanced",
        random_state=rng_seed,
    )

    pipe = ImbPipeline(steps=[
        ("clip", QuantileClipper(0.01, 0.99)),
        ("scale", RobustScaler()),
        ("smote", SMOTE(random_state=rng_seed, k_neighbors=int(smote_k_neighbors))),
        ("clf", clf),
    ])
    return pipe


def _minmax_01(scores: np.ndarray) -> np.ndarray:
    """
    Normaliza o score do teste para [0,1] (por fold).
    Isso permite usar threshold t em [0.1, 0.9] de forma consistente.
    """
    s = np.asarray(scores, dtype=float)
    s_min = float(np.min(s))
    s_max = float(np.max(s))
    return (s - s_min) / (s_max - s_min + 1e-12)


def pso_eval_svc_rbf(
    X,
    y,
    groups,
    log10C: float,
    log10gamma: float,
    t: float,
    n_splits: int = 5,
    rng_seed: int = 42,
    smote_k_neighbors_desired: int = 3,
):
    """
    Avaliação PSO multiobjetivo (2 objetivos):
        - f1 = bal_acc_mean  (MAX)
        - f2 = bal_acc_std   (MIN)

    Parâmetros do PSO (3):
        - log10C
        - log10gamma
        - t (threshold em [0.10, 0.90])

    Retorna: (bal_acc_mean, bal_acc_std)
    """
    # bounds defensivos
    t = float(np.clip(t, 0.10, 0.90))
    C = 10.0 ** float(log10C)
    gamma = 10.0 ** float(log10gamma)

    cv = GroupKFold(n_splits=n_splits)
    fold_bal_acc = []

    for tr_idx, te_idx in cv.split(X, y, groups=groups):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_te, y_te = X[te_idx], y[te_idx]

        # SMOTE seguro para este fold
        k_fold = _safe_smote_k(y_tr, desired_k=smote_k_neighbors_desired)

        pipe = _make_fold_pipeline_svc(
            C=C,
            gamma=gamma,
            rng_seed=rng_seed,
            smote_k_neighbors=k_fold
        )

        pipe.fit(X_tr, y_tr)

        # score contínuo do SVC (mais rápido que predict_proba)
        scores = pipe.decision_function(X_te)

        # normaliza para [0,1] e aplica threshold
        scores01 = _minmax_01(scores)
        y_pred = (scores01 >= t).astype(int)

        ba = balanced_accuracy_score(y_te, y_pred)
        fold_bal_acc.append(float(ba))

    bal_acc_mean = float(np.mean(fold_bal_acc))
    bal_acc_std = float(np.std(fold_bal_acc))

    return bal_acc_mean, bal_acc_std


In [ ]:
# =========================================================
# 1) Estruturas
# =========================================================

@dataclass
class Solution:
    x: np.ndarray              # posição: [log10C, log10gamma, t]
    f1: float                  # bal_acc_mean  (max)
    f2: float                  # bal_acc_std   (min)
    crowding: float = 0.0      # crowding distance


# Transformer para Outlier Handling: Quantile Clipping
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, low=0.01, high=0.99):
        self.low = low
        self.high = high

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lo_ = np.nanquantile(X, self.low, axis=0)
        self.hi_ = np.nanquantile(X, self.high, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lo_, self.hi_)

# =========================================================
# 2) Funções Pareto / Crowding
# =========================================================

def dominates(a: Solution, b: Solution) -> bool:
    """a domina b se f1>= e f2<= e pelo menos uma estrita."""
    return (a.f1 >= b.f1 and a.f2 <= b.f2) and (a.f1 > b.f1 or a.f2 < b.f2)


def non_dominated_set(solutions: List[Solution]) -> List[Solution]:
    """Retorna somente soluções não-dominadas."""
    nd = []
    for i, s in enumerate(solutions):
        dominated_flag = False
        for j, t in enumerate(solutions):
            if i != j and dominates(t, s):
                dominated_flag = True
                break
        if not dominated_flag:
            nd.append(s)
    return nd


def compute_crowding_distance(front: List[Solution]) -> None:
    """
    Crowding distance para 2 objetivos:
        - f1 (max) e f2 (min)
    Usamos normalização interna. Extremidades recebem inf.
    """
    if len(front) == 0:
        return
    if len(front) <= 2:
        for s in front:
            s.crowding = float("inf")
        return

    # inicializa
    for s in front:
        s.crowding = 0.0

    # Para f1 (max): ordena crescente
    front_sorted_f1 = sorted(front, key=lambda s: s.f1)
    f1_min = front_sorted_f1[0].f1
    f1_max = front_sorted_f1[-1].f1

    front_sorted_f1[0].crowding = float("inf")
    front_sorted_f1[-1].crowding = float("inf")

    denom = (f1_max - f1_min) if (f1_max - f1_min) > 1e-12 else 1.0
    for i in range(1, len(front_sorted_f1) - 1):
        if np.isinf(front_sorted_f1[i].crowding):
            continue
        prev_ = front_sorted_f1[i - 1].f1
        next_ = front_sorted_f1[i + 1].f1
        front_sorted_f1[i].crowding += (next_ - prev_) / denom

    # Para f2 (min): crowding calcula em eixo crescente também
    # mas como f2 é minimização, a ordenação ainda pode ser crescente.
    front_sorted_f2 = sorted(front, key=lambda s: s.f2)
    f2_min = front_sorted_f2[0].f2
    f2_max = front_sorted_f2[-1].f2

    front_sorted_f2[0].crowding = float("inf")
    front_sorted_f2[-1].crowding = float("inf")

    denom = (f2_max - f2_min) if (f2_max - f2_min) > 1e-12 else 1.0
    for i in range(1, len(front_sorted_f2) - 1):
        if np.isinf(front_sorted_f2[i].crowding):
            continue
        prev_ = front_sorted_f2[i - 1].f2
        next_ = front_sorted_f2[i + 1].f2
        front_sorted_f2[i].crowding += (next_ - prev_) / denom


def update_archive(
    archive: List[Solution],
    candidates: List[Solution],
    max_size: int = 50
) -> List[Solution]:
    """
    Atualiza o archive Pareto:
        - junta
        - filtra não-dominados
        - calcula crowding
        - corta pelo crowding se exceder max_size
    """
    combined = archive + candidates
    nd = non_dominated_set(combined)
    compute_crowding_distance(nd)

    if len(nd) <= max_size:
        return nd

    # mantém os mais "diversos"
    nd_sorted = sorted(nd, key=lambda s: s.crowding, reverse=True)
    return nd_sorted[:max_size]


def select_leader_tournament(archive: List[Solution], rng: np.random.Generator, k: int = 2) -> Solution:
    """
    Seleciona líder via torneio no archive, preferindo maior crowding distance.
    """
    if len(archive) == 0:
        raise ValueError("Archive vazio; não há líder para selecionar.")
    if len(archive) == 1:
        return archive[0]

    idx = rng.choice(len(archive), size=min(k, len(archive)), replace=False)
    pool = [archive[i] for i in idx]
    # maior crowding vence
    pool_sorted = sorted(pool, key=lambda s: s.crowding, reverse=True)
    return pool_sorted[0]


# =========================================================
# 3) Bounds, clamping e utilitários
# =========================================================

def clip_position(x: np.ndarray, bounds: np.ndarray) -> np.ndarray:
    """Clampa posição aos bounds. limita a particula dentro do espaço"""
    return np.minimum(np.maximum(x, bounds[:, 0]), bounds[:, 1])


def clamp_velocity(v: np.ndarray, vmax: np.ndarray) -> np.ndarray:
    """Clampa magnitude por dimensão.limita a velocidade das particulas"""
    return np.minimum(np.maximum(v, -vmax), vmax)


def eval_particle(
    x: np.ndarray,
    eval_fn: Callable[..., Tuple[float, float]],
    eval_kwargs: Dict[str, Any]
) -> Solution:
    """
    x = [log10C, log10gamma, t]
    eval_fn retorna: (bal_acc_mean, bal_acc_std)
    """
    f1, f2 = eval_fn(
        log10C=float(x[0]),
        log10gamma=float(x[1]),
        t=float(x[2]),
        **eval_kwargs
    )
    return Solution(x=x.copy(), f1=float(f1), f2=float(f2))


def choose_final_from_archive(
    archive: List[Solution],
    eps: float = 0.01
) -> Solution:
    """
    Regra de escolha final:
        1) pega soluções com f1 >= (f1_max - eps)
        2) entre elas, escolhe menor f2 (mais estável)
        3) empate: maior f1
    """
    if not archive:
        raise ValueError("Archive vazio.")

    f1_max = max(s.f1 for s in archive)
    top = [s for s in archive if s.f1 >= (f1_max - eps)]
    top_sorted = sorted(top, key=lambda s: (s.f2, -s.f1))
    return top_sorted[0]


# =========================================================
# 4) MOPSO principal
# =========================================================

def mopso_optimize(
    eval_fn: Callable[..., Tuple[float, float]],
    eval_kwargs: Dict[str, Any],
    n_particles: int = 25,
    n_iters: int = 35,
    archive_max: int = 50,
    seed: int = 42,
    # bounds: [log10C, log10gamma, t]
    bounds: np.ndarray = np.array([[-3.0, 3.0], [-6.0, 1.0], [0.10, 0.90]], dtype=float),
    # vmax por dimensão
    vmax: np.ndarray = np.array([0.5, 0.7, 0.08], dtype=float),
    # PSO coeficientes
    w_start: float = 0.9,
    w_end: float = 0.4,
    c1: float = 2.0,
    c2: float = 2.0,
    tournament_k: int = 2,
    early_stop_patience: Optional[int] = 8,
    early_stop_delta: float = 0.001,
    verbose: bool = True,
):
    """
    Retorna:
        - archive final (lista de Solution)
        - best_solution (Solution) escolhida pela regra topo+estabilidade
        - history (dict) com evolução do f1_max ao longo das iterações
    """
    rng = np.random.default_rng(seed)

    # inicializa posições e velocidades
    X = np.zeros((n_particles, 3), dtype=float)
    V = np.zeros((n_particles, 3), dtype=float)

    for i in range(n_particles):
        X[i, :] = rng.uniform(bounds[:, 0], bounds[:, 1])
        # velocidade inicial pequena
        V[i, :] = rng.uniform(-vmax, vmax) * 0.3

    # avalia partículas iniciais
    pbest = []
    current = []
    for i in range(n_particles):
        sol = eval_particle(X[i], eval_fn, eval_kwargs)
        current.append(sol)
        pbest.append(sol)

    archive: List[Solution] = []
    archive = update_archive(archive, current, max_size=archive_max)

    hist_f1_best = []
    best_f1_so_far = max(s.f1 for s in archive)
    hist_f1_best.append(best_f1_so_far)

    no_improve = 0

    for it in range(1, n_iters + 1):
        # inércia decrescente
        w = w_start + (w_end - w_start) * (it / n_iters)

        new_solutions = []

        for i in range(n_particles):
            leader = select_leader_tournament(archive, rng, k=tournament_k)

            r1 = rng.random(3)
            r2 = rng.random(3)

            # atualiza velocidade
            V[i] = (
                w * V[i]
                + c1 * r1 * (pbest[i].x - X[i])
                + c2 * r2 * (leader.x - X[i])
            )
            V[i] = clamp_velocity(V[i], vmax)

            # atualiza posição
            X[i] = X[i] + V[i]
            X[i] = clip_position(X[i], bounds)

            # avalia
            sol = eval_particle(X[i], eval_fn, eval_kwargs)
            new_solutions.append(sol)

            # atualiza pbest (regra multiobjetivo)
            if dominates(sol, pbest[i]):
                pbest[i] = sol
            elif dominates(pbest[i], sol):
                pass
            else:
                # não dominadas: desempate pela métrica rei (f1)
                if sol.f1 > pbest[i].f1:
                    pbest[i] = sol

        # atualiza archive
        archive = update_archive(archive, new_solutions, max_size=archive_max)

        # tracking e early stop
        f1_best = max(s.f1 for s in archive)
        hist_f1_best.append(f1_best)

        if verbose:
            best_now = choose_final_from_archive(archive, eps=0.01)
            print(
                f"[it {it:02d}/{n_iters}] "
                f"archive={len(archive):02d} "
                f"f1_max(bal_acc_mean)={f1_best:.4f} "
                f"best_choice: mean={best_now.f1:.4f}, std={best_now.f2:.4f}, x={best_now.x}"
            )

        if (f1_best - best_f1_so_far) >= early_stop_delta:
            best_f1_so_far = f1_best
            no_improve = 0
        else:
            no_improve += 1

        if early_stop_patience is not None and no_improve >= early_stop_patience:
            if verbose:
                print(f"Early stop: sem melhora >= {early_stop_delta} por {early_stop_patience} iterações.")
            break

    # escolha final
    best_solution = choose_final_from_archive(archive, eps=0.01)

    history = {
        "f1_best_per_iter": hist_f1_best,
        "iters_ran": len(hist_f1_best) - 1,
    }

    return archive, best_solution, history


# Execução do código

In [ ]:
df = pd.read_csv("dataset_reduced_top11.csv")

feature_cols = [c for c in df.columns if c not in ["status", "subject_id"]]

X = df[feature_cols].values
y = df["status"].astype(int).values
groups = df["subject_id"].values

In [ ]:
eval_kwargs = {
    "X": X,
    "y": y,
    "groups": groups,
    "n_splits": 5,
    "rng_seed": 42,
    "smote_k_neighbors_desired": 3,
}

In [ ]:
# chamada com valores menores (sensato para teste)
archive, best_sol, history = mopso_optimize(
    eval_fn=pso_eval_svc_rbf,
    eval_kwargs=eval_kwargs,
    n_particles=250,
    n_iters=500,
    archive_max=50,
    seed=42,
    early_stop_patience=300,
    early_stop_delta=0.001,
    verbose=True
)


# Convertendo para C*, gamma*, t*
log10C_star, log10g_star, t_star = best_sol.x
C_star = 10 ** float(log10C_star)
gamma_star = 10 ** float(log10g_star)
print(f'C_star: {C_star}; gamma_star: {gamma_star}; t_star:{t_star}')

In [ ]:


def _solution_to_row(sol):
    x = np.array(sol.x, dtype=float)
    log10C, log10gamma, t = x.tolist()
    C = 10 ** log10C
    gamma = 10 ** log10gamma
    return {
        "log10C": log10C,
        "log10gamma": log10gamma,
        "t": float(t),
        "C": float(C),
        "gamma": float(gamma),
        "bal_acc_mean": float(sol.f1),
        "bal_acc_std": float(sol.f2),
        "crowding": float(getattr(sol, "crowding", 0.0)),
    }


def export_mopso_artifacts(archive, best_sol, history, out_dir="mopso_results", top_n=15):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(out_dir, f"run_{ts}")
    os.makedirs(run_dir, exist_ok=True)

    # -------------------------
    # 1) Archive -> DataFrame
    # -------------------------
    df_archive = pd.DataFrame([_solution_to_row(s) for s in archive])
    # ranking: mean desc, std asc
    df_archive_sorted = df_archive.sort_values(
        ["bal_acc_mean", "bal_acc_std"], ascending=[False, True]
    ).reset_index(drop=True)

    archive_csv = os.path.join(run_dir, "archive_pareto.csv")
    archive_json = os.path.join(run_dir, "archive_pareto.json")
    df_archive_sorted.to_csv(archive_csv, index=False)
    df_archive_sorted.to_json(archive_json, orient="records", indent=2)

    # -------------------------
    # 2) History
    # -------------------------
    df_hist = pd.DataFrame({
        "iter": list(range(len(history["f1_best_per_iter"]))),
        "best_bal_acc_mean": history["f1_best_per_iter"],
    })
    hist_csv = os.path.join(run_dir, "history_convergence.csv")
    df_hist.to_csv(hist_csv, index=False)

    # -------------------------
    # 3) Best solution formatted
    # -------------------------
    best_row = _solution_to_row(best_sol)
    best_json = os.path.join(run_dir, "best_solution.json")
    best_txt = os.path.join(run_dir, "best_solution.txt")
    with open(best_json, "w", encoding="utf-8") as f:
        json.dump(best_row, f, indent=2)

    best_text = (
        "=== BEST SOLUTION (chosen) ===\n"
        f"log10C      : {best_row['log10C']:.6f}\n"
        f"log10gamma  : {best_row['log10gamma']:.6f}\n"
        f"C          : {best_row['C']:.8f}\n"
        f"gamma      : {best_row['gamma']:.10f}\n"
        f"threshold t: {best_row['t']:.4f}\n"
        f"bal_acc_mean: {best_row['bal_acc_mean']:.6f}\n"
        f"bal_acc_std : {best_row['bal_acc_std']:.6f}\n"
    )
    with open(best_txt, "w", encoding="utf-8") as f:
        f.write(best_text)

    # -------------------------
    # 4) Plot: Convergence
    # -------------------------
    fig = plt.figure()
    plt.plot(df_hist["iter"], df_hist["best_bal_acc_mean"])
    plt.xlabel("Iteration")
    plt.ylabel("Best Balanced Accuracy (mean)")
    plt.title("MOPSO Convergence (best bal_acc_mean per iteration)")
    plt.grid(True, alpha=0.3)
    conv_png = os.path.join(run_dir, "convergence.png")
    plt.savefig(conv_png, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    # -------------------------
    # 5) Plot: Pareto front (final archive)
    # -------------------------
    fig = plt.figure()
    plt.scatter(df_archive_sorted["bal_acc_std"], df_archive_sorted["bal_acc_mean"])
    plt.xlabel("Balanced Accuracy (std)  [minimize]")
    plt.ylabel("Balanced Accuracy (mean) [maximize]")
    plt.title("Final Pareto Front (Archive)")
    plt.grid(True, alpha=0.3)
    pareto_png = os.path.join(run_dir, "pareto_front.png")
    plt.savefig(pareto_png, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    # -------------------------
    # 6) Print: Top-N + best formatted
    # -------------------------
    print(f"\nSaved artifacts to: {run_dir}\n")
    print("=== TOP solutions (archive) ===")
    display(df_archive_sorted.head(top_n))

    print("\n" + best_text)

    return {
        "run_dir": run_dir,
        "archive_csv": archive_csv,
        "archive_json": archive_json,
        "history_csv": hist_csv,
        "best_json": best_json,
        "best_txt": best_txt,
        "convergence_png": conv_png,
        "pareto_png": pareto_png,
        "df_archive": df_archive_sorted,
        "df_history": df_hist,
        "best_row": best_row,
    }


# --- use after your PSO run ---
art = export_mopso_artifacts(archive, best_sol, history, out_dir="mopso_results", top_n=20)


In [ ]:
archive, best_sol, history = mopso_optimize(
    eval_fn=pso_eval_svc_rbf,
    eval_kwargs=eval_kwargs,
    n_particles=25,
    n_iters=35,
    archive_max=50,
    seed=42,
    verbose=True
)